# 01 — Explore
First pass over the 7 tables: shapes, types, nulls, odd values. Same question in pandas and DuckDB where it's useful.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import duckdb
from datawings import load_all, connect

pd.set_option("display.max_columns", 50)
dfs = load_all()
con = connect()
{k: v.shape for k, v in dfs.items()}

## Profile each table

In [ ]:
for name, df in dfs.items():
    print(f"\n== {name} {df.shape}")
    print(df.isna().sum()[lambda s: s > 0].to_string() or 'no nulls')

## pandas vs DuckDB — warm-up
Flights by status, both ways.

In [ ]:
dfs["flight"]["status"].value_counts()

In [ ]:
con.sql("""
    SELECT status, count(*) AS n
    FROM stg_flight
    GROUP BY ALL
    ORDER BY n DESC
""").df()

DuckDB can also query a pandas DataFrame in scope directly:

In [ ]:
crew = dfs["crew"]
duckdb.sql("SELECT rank, count(*) n FROM crew GROUP BY ALL ORDER BY n DESC").df()

## Questions to chase
- How clean is `crew.rank`? What's the mapping?
- Does the qualification `status` flag agree with the dates?
- Actual vs planned FDP — how many duties breach the MAX_FDP limit?
- Rolling 7d duty / 28d flight hours per crew — who gets close to the limits?
- Disruptions: what triggers them, how are they recovered, what does it cost in delay?